In [1]:
import pandas as pd

In [119]:
sheet_url = 'https://docs.google.com/spreadsheets/d/1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk/export?format=csv&gid=2141931777'
sheet_url2 = 'https://docs.google.com/spreadsheets/d/1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE/export?format=csv&gid=2141931777'

df = pd.read_csv(sheet_url)

In [120]:
# Rename columns.
df.columns = ['P1','P2','P1_WINS','P2_WINS','WINNER1','WINNER2','P1_ARCH','P2_ARCH','P1_SUBARCH','P2_SUBARCH','P1_NOTE','P2_NOTE','DATE','EVENT_TYPE']

# Replace null values with 'NA' string.
df.fillna({'P1_ARCH':'NA','P2_ARCH':'NA','P1_SUBARCH':'NA','P2_SUBARCH':'NA','P1_NOTE':'NA','P2_NOTE':'NA'},inplace=True)

# Strip whitespace from player/deck names.
df.P1 = df.P1.str.strip()
df.P2 = df.P2.str.strip()
df.P1_ARCH = df.P1_ARCH.str.strip()
df.P2_ARCH = df.P2_ARCH.str.strip()
df.P1_SUBARCH = df.P1_SUBARCH.str.strip()
df.P2_SUBARCH = df.P2_SUBARCH.str.strip()
df.P1_NOTE = df.P1_NOTE.str.strip()
df.P2_NOTE = df.P2_NOTE.str.strip()

# Format EVENT_TYPE values.
df['EVENT_TYPE'] = df['EVENT_TYPE'].str.upper().str.strip()

# Format No Show deck name values.
for index, row in df.iterrows():
    if row['P1_SUBARCH'].strip().upper() == 'NO SHOW':
        df.at[index,'P1_SUBARCH'] = 'NO SHOW'
    if row['P2_SUBARCH'].strip().upper() == 'NO SHOW':
        df.at[index,'P2_SUBARCH'] = 'NO SHOW'

# Format date column.
df.DATE = pd.to_datetime(df.DATE,yearfirst=False,format='mixed')

# Adding Event_IDs.
count = 1000001
df['EVENT_ID'] = 0
for index, row in reversed(list(df.iterrows())):
    df.at[index,'EVENT_ID'] = count
    if pd.notna(row['EVENT_TYPE']):
        count += 1

# Handle empty EVENT_TYPE/DATE values by forward-filling.
df['EVENT_TYPE'] = df['EVENT_TYPE'].ffill()
df['DATE'] = df['DATE'].ffill()

# Ignore events with incomplete data.
events_to_ignore = [1000007]
df = df[~df.EVENT_ID.isin(pd.to_datetime(events_to_ignore))]

# Replace Winner1/2 columns with single Match_Winner column.
df['MATCH_WINNER'] = df.apply(lambda row: 'P1' if row['P1_WINS'] > row['P2_WINS'] else ('P2' if row['P2_WINS'] > row['P1_WINS'] else 'NA'), axis=1)
df.drop(columns=['WINNER1','WINNER2'],inplace=True)

# EVENT_ID 1000067 should be OTHER
df.loc[df['EVENT_ID'] == 1000067,'EVENT_TYPE'] = 'OTHER'



In [121]:
df.to_excel('output.xlsx', index=False)

In [106]:
df

,P1,P2,P1_Wins,P2_Wins,P1_Arch,P2_Arch,P1_Subarch,P2_Subarch,P1_Deck,P2_Deck,Date,Event_Type,Event_ID,Match_Winner
0,RedandSnap1,Mogged,1.0,0.0,Bazaar,Shops,Dredge,Sphere Shops,NA,NA,2025-01-10,CHALLENGE,1019074.0,P1
1,Mogged,RedandSnap1,0.0,1.0,Shops,Bazaar,Sphere Shops,Dredge,NA,NA,2025-01-10,CHALLENGE,1019073.0,P2
2,RedandSnap1,wolf2222,1.0,0.0,Bazaar,Bazaar,Dredge,Dredge,NA,NA,2025-01-10,CHALLENGE,1019072.0,P1
3,Mogged,albertoSD,1.0,0.0,Shops,Aggro,Sphere Shops,Initiative,NA,NA,2025-01-10,CHALLENGE,1019071.0,P1
4,wolf2222,RedandSnap1,0.0,1.0,Bazaar,Bazaar,Dredge,Dredge,NA,NA,2025-01-10,CHALLENGE,1019070.0,P2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19069,2plus2isfive,Zenith777,1.0,2.0,Oath,Bazaar,Oath,Dredge,NA,NA,2024-08-29,CHALLENGE,1000005.0,P2
19070,s063,unluckymonkey,0.0,2.0,Combo,Lurrus,Doomsday,Lurrus DRS,NA,NA,2024-08-29,CHALLENGE,1000004.0,P2
19071,OrnatePuzzles,TonyScapone,1.0,2.0,Shops,Shops,Sphere Shops,Jewel Shops,NA,NA,2024-08-29,CHALLENGE,1000003.0,P2
19072,Ale_Mtg,AFX,0.0,2.0,Fair Blue,Bazaar,BUG,Countervine,NA,NA,2024-08-29,CHALLENGE,1000002.0,P2


In [100]:
df_decks = df.groupby(['P1_Arch','P1_Subarch']).agg({'P1_Deck':'count'}).reset_index().sort_values(by=['P1_Arch'])
df_decks['DECK_FORMAT'] = 'VINTAGE'
df_decks['DECK_ID'] = 0

# Adding DECK_IDs.
count = 1001
for index, row in df_decks.iterrows():
    df_decks.at[index,'DECK_ID'] = count
    count += 1

df_decks.drop(columns=['P1_Deck'],inplace=True)
df_decks

,P1_Arch,P1_Subarch,DECK_FORMAT,DECK_ID
0,Aggro,Eldrazi,VINTAGE,1001
1,Aggro,Initiative,VINTAGE,1002
2,Aggro,Merfolk,VINTAGE,1003
3,Aggro,Other Aggro,VINTAGE,1004
4,Aggro,Red Prison,VINTAGE,1005
7,Bazaar,Countervine,VINTAGE,1006
8,Bazaar,Dredge,VINTAGE,1007
5,Bazaar,AggroVine,VINTAGE,1008
6,Bazaar,Aggrovine,VINTAGE,1009
15,Combo,Tinker,VINTAGE,1010


In [73]:
df.to_excel('output.xlsx', index=False)

In [44]:
df.P1_Arch.isna().sum()

5

In [49]:
df.fillna({'P1_Arch':'NA','P2_Arch':'NA','P1_Subarch':'NA','P2_Subarch':'NA','P1_Deck':'NA','P2_Deck':'NA'},inplace=True)

In [51]:
df[df['P1_Wins'].isna()]

,P1,P2,P1_Wins,P2_Wins,Winner1,Winner2,P1_Arch,P2_Arch,P1_Subarch,P2_Subarch,P1_Deck,P2_Deck,Date,Event_Type,Match_Winner
16898,laughingrock,ecobaronen,NaN,NaN,1,0,Bazaar,Shops,Dredge,Jewel Shops,NA,NA,2024-09-06,challenge,NA
16899,ecobaronen,laughingrock,NaN,NaN,0,1,Shops,Bazaar,Jewel Shops,Dredge,NA,NA,2024-09-06,NaN,NA
16900,laughingrock,JackSparr0w,NaN,NaN,1,0,Bazaar,Shops,Dredge,Jewel Shops,NA,NA,2024-09-06,NaN,NA
16901,ecobaronen,Montolio,NaN,NaN,1,0,Shops,Lurrus,Jewel Shops,Esper Lurrus Control,NA,NA,2024-09-06,NaN,NA
16902,Montolio,ecobaronen,NaN,NaN,0,1,Lurrus,Shops,Esper Lurrus Control,Jewel Shops,NA,NA,2024-09-06,NaN,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17117,NishaTelya,Xuxa,NaN,NaN,0,1,Combo,Bazaar,PO,Dredge,NA,NA,2024-09-06,NaN,NA
17118,albertoSD,death_grips,NaN,NaN,0,1,Aggro,Aggro,Initiative,Initiative,NA,NA,2024-09-06,NaN,NA
17119,hodortimebaby,KingHairy,NaN,NaN,0,1,Bazaar,Oath,Dredge,Oath,NA,NA,2024-09-06,NaN,NA
17120,jben,LucasG1ggs,NaN,NaN,0,1,Lurrus,Bazaar,Grixis Lurrus Control,Dredge,NA,NA,2024-09-06,NaN,NA
